In [1]:
import pandas as pd


df = pd.read_parquet('yellow_tripdata_2021-01.csv.parquet')

print(df.head(100))
print(df.dtypes)

    VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0          1  2021-01-01 00:30:10   2021-01-01 00:36:12              1.0   
1          1  2021-01-01 00:51:20   2021-01-01 00:52:19              1.0   
2          1  2021-01-01 00:43:30   2021-01-01 01:11:06              1.0   
3          1  2021-01-01 00:15:48   2021-01-01 00:31:01              0.0   
4          2  2021-01-01 00:31:49   2021-01-01 00:48:21              1.0   
..       ...                  ...                   ...              ...   
95         2  2021-01-01 00:12:41   2021-01-01 00:26:47              1.0   
96         2  2021-01-01 00:23:29   2021-01-01 00:35:03              2.0   
97         2  2021-01-01 00:46:17   2021-01-01 00:54:25              2.0   
98         2  2021-01-01 00:28:16   2021-01-01 00:51:44              1.0   
99         2  2021-01-01 00:42:35   2021-01-01 00:54:41              3.0   

    trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  \
0      

In [2]:
import pyarrow.parquet as pq

file_path = "yellow_tripdata_2021-01.csv.parquet"

parquet_file = pq.ParquetFile(file_path)

for batch in parquet_file.iter_batches(batch_size=100_000):
    chunk = batch.to_pandas()
    print(chunk.shape)

(100000, 19)
(100000, 19)
(100000, 19)
(100000, 19)
(100000, 19)
(100000, 19)
(100000, 19)
(100000, 19)
(100000, 19)
(100000, 19)
(100000, 19)
(100000, 19)
(100000, 19)
(69769, 19)


In [3]:
print(df.head())
print(df.dtypes)

   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         1  2021-01-01 00:30:10   2021-01-01 00:36:12              1.0   
1         1  2021-01-01 00:51:20   2021-01-01 00:52:19              1.0   
2         1  2021-01-01 00:43:30   2021-01-01 01:11:06              1.0   
3         1  2021-01-01 00:15:48   2021-01-01 00:31:01              0.0   
4         2  2021-01-01 00:31:49   2021-01-01 00:48:21              1.0   

   trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  \
0           2.10         1.0                  N           142            43   
1           0.20         1.0                  N           238           151   
2          14.70         1.0                  N           132           165   
3          10.60         1.0                  N           138           132   
4           4.94         1.0                  N            68            33   

   payment_type  fare_amount  extra  mta_tax  tip_amount  tolls_amount  \


In [4]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Column names:", df.columns.tolist())

Rows: 1369769
Columns: 19
Column names: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee']


In [5]:
import pyarrow.parquet as pq

parquet_file = pq.ParquetFile(file_path)

print("Total rows:", parquet_file.metadata.num_rows)
print("Number of columns:", parquet_file.metadata.num_columns)
print("Row groups:", parquet_file.num_row_groups)

Total rows: 1369769
Number of columns: 19
Row groups: 1


In [6]:
from collections import Counter
null_counts = Counter()
total_rows = 0
minimums = {}
maximums = {}
column_types = {}

for batch in parquet_file.iter_batches(batch_size=100_000):
    chunk = batch.to_pandas()
    total_rows += len(chunk)

    for col in chunk.columns:
        column_types[col] = str(chunk[col].dtype)
        null_counts[col] += int(chunk[col].isna().sum())

        values = chunk[col].dropna()

        if values.empty:
            continue

        if pd.api.types.is_numeric_dtype(values):
            current_min = values.min()
            current_max = values.max()
        elif pd.api.types.is_datetime64_any_dtype(values):
            current_min = values.min()
            current_max = values.max()
        else:
            continue

        if col not in minimums or current_min < minimums[col]:
            minimums[col] = current_min

        if col not in maximums or current_max > maximums[col]:
            maximums[col] = current_max

profile = pd.DataFrame({
    "dtype": pd.Series(column_types),
    "null_count": pd.Series(dict(null_counts)),
})

profile["null_percentage"] = (
    profile["null_count"] / total_rows * 100
).round(2)

profile["min"] = pd.Series(minimums)
profile["max"] = pd.Series(maximums)

print(profile.to_string())
print("Total rows:", total_rows)

                                dtype  null_count  null_percentage                  min                  max
VendorID                        int64           0             0.00                    1                    6
tpep_pickup_datetime   datetime64[us]           0             0.00  2008-12-31 23:05:14  2021-02-22 16:52:16
tpep_dropoff_datetime  datetime64[us]           0             0.00  2008-12-31 23:07:22  2021-02-22 16:56:15
passenger_count               float64       98352             7.18                  0.0                  8.0
trip_distance                 float64           0             0.00                  0.0            263163.28
RatecodeID                    float64       98352             7.18                  1.0                 99.0
store_and_fwd_flag             object       98352             7.18                  NaN                  NaN
PULocationID                    int64           0             0.00                    1                  265
DOLocationID       

In [7]:
from collections import Counter
problem_counts = Counter()
seen_hashes = set()

for batch in parquet_file.iter_batches(batch_size=100_000):
    chunk = batch.to_pandas()

    pickup = pd.to_datetime(
        chunk["tpep_pickup_datetime"], errors="coerce"
    )

    outside_january = (
        pickup.isna()
        | (pickup < "2021-01-01")
        | (pickup >= "2021-02-01")
    )

    problem_counts["pickup_outside_january_or_invalid"] += int(
        outside_january.sum()
    )

    for col in ["fare_amount", "trip_distance"]:
        values = pd.to_numeric(chunk[col], errors="coerce")

        problem_counts[f"{col}_negative"] += int((values < 0).sum())
        problem_counts[f"{col}_zero"] += int((values == 0).sum())

    problem_counts["passenger_count_zero"] += int(
        (chunk["passenger_count"] == 0).sum()
    )

    row_hashes = pd.util.hash_pandas_object(
        chunk, index=False
    )

    for h in row_hashes:
        h = int(h)
        if h in seen_hashes:
            problem_counts["duplicate_rows"] += 1
        else:
            seen_hashes.add(h)

problems = pd.DataFrame(
    list(problem_counts.items()),
    columns=["Problem", "Count"]
)

print(problems.to_string(index=False))

                          Problem  Count
pickup_outside_january_or_invalid     24
             fare_amount_negative   6769
                 fare_amount_zero    642
           trip_distance_negative      0
               trip_distance_zero  19952
             passenger_count_zero  26726


In [8]:
def clean(df, seen_hashes=None):
    cleaned = df.copy()

    removed = {
        "invalid_or_outside_january": 0,
        "invalid_fare": 0,
        "invalid_distance": 0,
        "zero_passengers": 0,
        "duplicates": 0
    }

    # 1. Remove invalid pickup dates and dates outside January
    pickup = pd.to_datetime(
        cleaned["tpep_pickup_datetime"],
        errors="coerce"
    )

    mask = (
        pickup.notna()
        & (pickup >= "2021-01-01")
        & (pickup < "2021-02-01")
    )

    removed["invalid_or_outside_january"] = int((~mask).sum())
    cleaned = cleaned.loc[mask].copy()

    # 2. Remove invalid fares
    fare = pd.to_numeric(cleaned["fare_amount"], errors="coerce")
    mask = fare.notna() & (fare > 0)

    removed["invalid_fare"] = int((~mask).sum())
    cleaned = cleaned.loc[mask].copy()

    # 3. Remove invalid trip distances
    distance = pd.to_numeric(
        cleaned["trip_distance"], errors="coerce"
    )
    mask = distance.notna() & (distance > 0)

    removed["invalid_distance"] = int((~mask).sum())
    cleaned = cleaned.loc[mask].copy()

    # 4. Remove zero or missing passenger counts
    passengers = pd.to_numeric(
        cleaned["passenger_count"], errors="coerce"
    )
    mask = passengers.notna() & (passengers != 0)

    removed["zero_passengers"] = int((~mask).sum())
    cleaned = cleaned.loc[mask].copy()

    # 5. Remove duplicate rows
    row_hashes = pd.util.hash_pandas_object(
        cleaned, index=False
    )

    if seen_hashes is None:
        duplicate_mask = cleaned.duplicated(keep="first")
        removed["duplicates"] = int(duplicate_mask.sum())
        cleaned = cleaned.loc[~duplicate_mask].copy()
    else:
        keep = []

        for h in row_hashes:
            h = int(h)
            if h in seen_hashes:
                keep.append(False)
                removed["duplicates"] += 1
            else:
                keep.append(True)
                seen_hashes.add(h)

        cleaned = cleaned.loc[keep].copy()

    return cleaned, removed

In [27]:
from collections import Counter

cleaning_counts = Counter()
rows_before = len(df)
rows_after = len(df)
print("Check row counts:")
print("Before:", rows_before)
print("After:", rows_after)
print("Difference:", rows_before - rows_after)

print("\nCleaning rules:")
for rule, count in cleaning_counts.items():
    print(rule, ":", count)

Check row counts:
Before: 1369769
After: 1369769
Difference: 0

Cleaning rules:


In [9]:
first_batch = next(
    parquet_file.iter_batches(batch_size=100)
)

sample = first_batch.to_pandas()

print(sample.head())
print(sample.dtypes)

   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         1  2021-01-01 00:30:10   2021-01-01 00:36:12              1.0   
1         1  2021-01-01 00:51:20   2021-01-01 00:52:19              1.0   
2         1  2021-01-01 00:43:30   2021-01-01 01:11:06              1.0   
3         1  2021-01-01 00:15:48   2021-01-01 00:31:01              0.0   
4         2  2021-01-01 00:31:49   2021-01-01 00:48:21              1.0   

   trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  \
0           2.10         1.0                  N           142            43   
1           0.20         1.0                  N           238           151   
2          14.70         1.0                  N           132           165   
3          10.60         1.0                  N           138           132   
4           4.94         1.0                  N            68            33   

   payment_type  fare_amount  extra  mta_tax  tip_amount  tolls_amount  \


In [10]:
from collections import Counter

seen_hashes = set()
cleaning_counts = Counter()
rows_before = parquet_file.metadata.num_rows
rows_after = 0

for batch in parquet_file.iter_batches(batch_size=100_000):
    chunk = batch.to_pandas()

    cleaned_chunk, removed_counts = clean(
        chunk,
        seen_hashes=seen_hashes
    )

    cleaning_counts.update(removed_counts)
    rows_after += len(cleaned_chunk)

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Rows removed:", rows_before - rows_after)

print("\nCleaning rules:")
for rule, count in cleaning_counts.items():
    print(f"{rule}: {count}")

Rows before: 1369769
Rows after: 1220108
Rows removed: 149661

Cleaning rules:
invalid_or_outside_january: 24
invalid_fare: 7410
invalid_distance: 18968
zero_passengers: 123259
duplicates: 0


In [11]:

print("Final Data Validation")
print("-" * 30)

print("Rows before cleaning:", rows_before)
print("Rows after cleaning:", len(df))
print("Rows removed:", rows_before - len(df))
print("Duplicate rows remaining:", df.duplicated().sum())
print("Missing values:", df.isnull().sum().sum())

print("\nData types:")
print(df.dtypes)

print("\nFirst 5 cleaned rows:")
print(df.head())


Final Data Validation
------------------------------
Rows before cleaning: 1369769
Rows after cleaning: 1369769
Rows removed: 0
Duplicate rows remaining: 0
Missing values: 1763172

Data types:
VendorID                          int64
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag               object
PULocationID                      int64
DOLocationID                      int64
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
airport_fee                     float64
dtype: object

First 5 cleaned rows:
   VendorI

In [12]:

print("DataFrame shape:", df.shape)
print("DataFrame columns:", df.columns.tolist())
print("Missing values per column:")
print(df.isnull().sum())


DataFrame shape: (1369769, 19)
DataFrame columns: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee']
Missing values per column:
VendorID                       0
tpep_pickup_datetime           0
tpep_dropoff_datetime          0
passenger_count            98352
trip_distance                  0
RatecodeID                 98352
store_and_fwd_flag         98352
PULocationID                   0
DOLocationID                   0
payment_type                   0
fare_amount                    0
extra                          0
mta_tax                        0
tip_amount                     0
tolls_amount                   0
improvement_surcharge          0
total_amount                   0
congestion_surcharge       98352
ai

In [13]:

import pandas as pd

df_clean = df.copy()

rows_before = len(df_clean)
cleaning_counts = {}

invalid_date = (
    df_clean["tpep_pickup_datetime"].isna()
    | df_clean["tpep_dropoff_datetime"].isna()
    | (df_clean["tpep_pickup_datetime"] < pd.Timestamp("2021-01-01"))
    | (df_clean["tpep_pickup_datetime"] >= pd.Timestamp("2021-02-01"))
    | (df_clean["tpep_dropoff_datetime"] < df_clean["tpep_pickup_datetime"])
)

cleaning_counts["invalid_or_outside_january"] = int(invalid_date.sum())
df_clean = df_clean.loc[~invalid_date].copy()

invalid_fare = df_clean["fare_amount"] <= 0
cleaning_counts["invalid_fare"] = int(invalid_fare.sum())
df_clean = df_clean.loc[~invalid_fare].copy()

invalid_distance = df_clean["trip_distance"] <= 0
cleaning_counts["invalid_distance"] = int(invalid_distance.sum())
df_clean = df_clean.loc[~invalid_distance].copy()

zero_passengers = df_clean["passenger_count"] <= 0
cleaning_counts["zero_passengers"] = int(zero_passengers.sum())
df_clean = df_clean.loc[~zero_passengers].copy()

duplicates = df_clean.duplicated()
cleaning_counts["duplicates"] = int(duplicates.sum())
df_clean = df_clean.loc[~duplicates].copy()

rows_after = len(df_clean)

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Rows removed:", rows_before - rows_after)
print("\nCleaning rules:")
for rule, count in cleaning_counts.items():
    print(f"{rule}: {count}")

print("\nCleaned DataFrame shape:", df_clean.shape)


Rows before: 1369769
Rows after: 1311496
Rows removed: 58273

Cleaning rules:
invalid_or_outside_january: 5666
invalid_fare: 7394
invalid_distance: 18967
zero_passengers: 26246
duplicates: 0

Cleaned DataFrame shape: (1311496, 19)


In [14]:

print("Final shape:", df_clean.shape)
print("Duplicate rows:", df_clean.duplicated().sum())
print("Missing values:", df_clean.isnull().sum().sum())
print(df_clean.head())


Final shape: (1311496, 19)
Duplicate rows: 0
Missing values: 1677047
   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         1  2021-01-01 00:30:10   2021-01-01 00:36:12              1.0   
1         1  2021-01-01 00:51:20   2021-01-01 00:52:19              1.0   
2         1  2021-01-01 00:43:30   2021-01-01 01:11:06              1.0   
4         2  2021-01-01 00:31:49   2021-01-01 00:48:21              1.0   
5         1  2021-01-01 00:16:29   2021-01-01 00:24:30              1.0   

   trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  \
0           2.10         1.0                  N           142            43   
1           0.20         1.0                  N           238           151   
2          14.70         1.0                  N           132           165   
4           4.94         1.0                  N            68            33   
5           1.60         1.0                  N           224            68   

   pa

In [15]:

print("Cleaning summary")
print("=" * 35)

for rule, count in cleaning_counts.items():
    print(f"{rule}: {count:,}")

print("=" * 35)
print(f"Before: {rows_before:,}")
print(f"After: {len(df_clean):,}")
print(f"Removed: {rows_before - len(df_clean):,}")


Cleaning summary
invalid_or_outside_january: 5,666
invalid_fare: 7,394
invalid_distance: 18,967
zero_passengers: 26,246
duplicates: 0
Before: 1,369,769
After: 1,311,496
Removed: 58,273


In [16]:

print("Passenger count:")
print(df["passenger_count"].value_counts(dropna=False).head(10))

print("\nFare amount:")
print(df["fare_amount"].describe())

print("\nTrip distance:")
print(df["trip_distance"].describe())

print("\nPickup dates:")
print(df["tpep_pickup_datetime"].min())
print(df["tpep_pickup_datetime"].max())

print("\nPassenger count missing:")
print(df["passenger_count"].isna().sum())


Passenger count:
passenger_count
1.0    966236
2.0    161671
NaN     98352
3.0     43935
5.0     31089
0.0     26726
6.0     25362
4.0     16391
7.0         5
8.0         2
Name: count, dtype: int64

Fare amount:
count    1.369769e+06
mean     1.209663e+01
std      1.291337e+01
min     -4.900000e+02
25%      6.000000e+00
50%      8.500000e+00
75%      1.350000e+01
max      6.960500e+03
Name: fare_amount, dtype: float64

Trip distance:
count    1.369769e+06
mean     4.631983e+00
std      3.939037e+02
min      0.000000e+00
25%      1.000000e+00
50%      1.700000e+00
75%      3.020000e+00
max      2.631633e+05
Name: trip_distance, dtype: float64

Pickup dates:
2008-12-31 23:05:14
2021-02-22 16:52:16

Passenger count missing:
98352


In [17]:

print("Validation on original data")
print("=" * 40)

pickup = df["tpep_pickup_datetime"]
dropoff = df["tpep_dropoff_datetime"]

checks = {
    "Pickup outside January": (
        (pickup < "2021-01-01") |
        (pickup >= "2021-02-01")
    ).sum(),
    "Dropoff before pickup": (
        dropoff < pickup
    ).sum(),
    "Invalid fare (<= 0)": (
        df["fare_amount"] <= 0
    ).sum(),
    "Invalid distance (<= 0)": (
        df["trip_distance"] <= 0
    ).sum(),
    "Zero passengers": (
        df["passenger_count"] == 0
    ).sum(),
    "Missing passengers": (
        df["passenger_count"].isna()
    ).sum()
}

for rule, count in checks.items():
    print(f"{rule}: {count:,}")


Validation on original data
Pickup outside January: 24
Dropoff before pickup: 5,642
Invalid fare (<= 0): 7,411
Invalid distance (<= 0): 19,952
Zero passengers: 26,726
Missing passengers: 98,352


In [18]:

import pandas as pd

df_clean = df.copy()

rows_before = len(df_clean)

pickup = df_clean["tpep_pickup_datetime"]
dropoff = df_clean["tpep_dropoff_datetime"]

invalid_date = (
    pickup.isna()
    | dropoff.isna()
    | (pickup < pd.Timestamp("2021-01-01"))
    | (pickup >= pd.Timestamp("2021-02-01"))
    | (dropoff < pickup)
)

invalid_fare = df_clean["fare_amount"] <= 0
invalid_distance = df_clean["trip_distance"] <= 0

invalid_passengers = (
    df_clean["passenger_count"].isna()
    | (df_clean["passenger_count"] <= 0)
)

invalid_rows = (
    invalid_date
    | invalid_fare
    | invalid_distance
    | invalid_passengers
)

df_clean = df_clean.loc[~invalid_rows].copy()

print("Rows before:", rows_before)
print("Rows after:", len(df_clean))
print("Rows removed:", rows_before - len(df_clean))
print("Duplicate rows remaining:", df_clean.duplicated().sum())
print("Missing values remaining:", df_clean.isnull().sum().sum())


Rows before: 1369769
Rows after: 1220107
Rows removed: 149662
Duplicate rows remaining: 0
Missing values remaining: 1220102


In [19]:

output_file = "yellow_tripdata_2021-01_cleaned.parquet"

df_clean.to_parquet(output_file, index=False)

print("Cleaned dataset saved successfully:", output_file)
print("Final shape:", df_clean.shape)


Cleaned dataset saved successfully: yellow_tripdata_2021-01_cleaned.parquet
Final shape: (1220107, 19)


In [20]:

df_check = pd.read_parquet(output_file)

print("Saved file shape:", df_check.shape)
print("Duplicate rows:", df_check.duplicated().sum())
print("Missing values:", df_check.isnull().sum().sum())

print("Pickup minimum:", df_check["tpep_pickup_datetime"].min())
print("Pickup maximum:", df_check["tpep_pickup_datetime"].max())
print("Minimum fare:", df_check["fare_amount"].min())
print("Minimum distance:", df_check["trip_distance"].min())
print("Minimum passengers:", df_check["passenger_count"].min())


Saved file shape: (1220107, 19)
Duplicate rows: 0
Missing values: 1220102
Pickup minimum: 2021-01-01 00:00:04
Pickup maximum: 2021-01-31 23:59:49
Minimum fare: 0.01
Minimum distance: 0.01
Minimum passengers: 1.0


In [21]:

missing = df_check.isnull().sum()

missing_report = pd.DataFrame({
    "Missing_Count": missing,
    "Missing_Percentage": (
        missing / len(df_check) * 100
    ).round(2)
})

missing_report = missing_report[
    missing_report["Missing_Count"] > 0
].sort_values("Missing_Count", ascending=False)

print(missing_report)


             Missing_Count  Missing_Percentage
airport_fee        1220102               100.0


In [ ]:

df_check = df_check.dropna(axis=1, how="all")

print("Final shape:", df_check.shape)
print("Missing values:", df_check.isnull().sum().sum())
print("Remaining columns:")
print(df_check.columns.tolist())


Final shape: (1220107, 19)
Missing values: 1220102
Remaining columns:
['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee']


In [24]:

print("Non-missing values:")
print(df_check["airport_fee"].notna().sum())

print("\nValues including missing:")
print(df_check["airport_fee"].value_counts(dropna=False))

print("\nNon-null sample:")
print(df_check.loc[df_check["airport_fee"].notna(), ["airport_fee"]].head(10))


Non-missing values:
5

Values including missing:
airport_fee
NaN    1220102
0.0          5
Name: count, dtype: int64

Non-null sample:
        airport_fee
151172          0.0
153339          0.0
154717          0.0
154718          0.0
430109          0.0


In [25]:

df_check = df_check.drop(columns=["airport_fee"])

print("Final shape:", df_check.shape)
print("Missing values:", df_check.isnull().sum().sum())
print("Columns:", df_check.columns.tolist())


Final shape: (1220107, 18)
Missing values: 0
Columns: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge']


In [26]:

df_check.to_parquet(
    "yellow_tripdata_2021-01_final.parquet",
    index=False
)


In [27]:

df_final = df_clean.copy()

df_final = df_final.drop(columns=["airport_fee"])

print("Final shape:", df_final.shape)
print("Missing values:", df_final.isnull().sum().sum())
print("Duplicate rows:", df_final.duplicated().sum())

print("Pickup minimum:", df_final["tpep_pickup_datetime"].min())
print("Pickup maximum:", df_final["tpep_pickup_datetime"].max())
print("Minimum fare:", df_final["fare_amount"].min())
print("Minimum distance:", df_final["trip_distance"].min())
print("Minimum passengers:", df_final["passenger_count"].min())


Final shape: (1220107, 18)
Missing values: 0
Duplicate rows: 0
Pickup minimum: 2021-01-01 00:00:04
Pickup maximum: 2021-01-31 23:59:49
Minimum fare: 0.01
Minimum distance: 0.01
Minimum passengers: 1.0


In [28]:

output_file = "yellow_tripdata_2021-01_final.parquet"

df_final.to_parquet(output_file, index=False)

df_saved = pd.read_parquet(output_file)

assert df_saved.shape == df_final.shape
assert df_saved.duplicated().sum() == 0
assert df_saved["passenger_count"].notna().all()
assert (df_saved["passenger_count"] > 0).all()
assert (df_saved["fare_amount"] > 0).all()
assert (df_saved["trip_distance"] > 0).all()

print("All validation checks passed!")
print("Saved file:", output_file)
print("Final shape:", df_saved.shape)


All validation checks passed!
Saved file: yellow_tripdata_2021-01_final.parquet
Final shape: (1220107, 18)
